In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


class BrainMaskExtractor:

    @staticmethod
    def extract_brain_mask(image, threshold=10, morph_kernel_size=5):
       
        if isinstance(image, Image.Image):
            image = np.array(image)

        # Convert to grayscale if RGB
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image

        # Threshold to separate brain from background
        _, brain_mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

        # Morphological operations to clean up mask
        kernel = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
        brain_mask = cv2.morphologyEx(brain_mask, cv2.MORPH_CLOSE, kernel)
        brain_mask = cv2.morphologyEx(brain_mask, cv2.MORPH_OPEN, kernel)

        # Find largest contour (brain region)
        contours, _ = cv2.findContours(brain_mask, cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            brain_mask = np.zeros_like(brain_mask)
            cv2.drawContours(brain_mask, [largest_contour], -1, 255, -1)

        return brain_mask

    @staticmethod
    def get_brain_bbox(brain_mask):
        coords = cv2.findNonZero(brain_mask)
        if coords is not None:
            x, y, w, h = cv2.boundingRect(coords)
            return x, y, w, h
        return None

    @staticmethod
    def crop_to_brain(image, brain_mask, padding=10):
      
        bbox = BrainMaskExtractor.get_brain_bbox(brain_mask)
        if bbox is None:
            return image

        x, y, w, h = bbox
        h_img, w_img = image.shape[:2]

        # Add padding
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(w_img, x + w + padding)
        y2 = min(h_img, y + h + padding)

        if len(image.shape) == 3:
            return image[y1:y2, x1:x2, :]
        return image[y1:y2, x1:x2]

    @staticmethod
    def apply_mask(image, mask):
       
        if isinstance(image, Image.Image):
            image = np.array(image)
        if isinstance(mask, Image.Image):
            mask = np.array(mask)

        mask = (mask > 127).astype(np.uint8) * 255

        if len(image.shape) == 3:
            mask_3channel = np.stack([mask] * 3, axis=-1)
            masked_image = cv2.bitwise_and(image, mask_3channel)
        else:
            masked_image = cv2.bitwise_and(image, mask)

        return masked_image

class BrainTumorDataset(Dataset):


    def __init__(self, root_dir, transform=None, use_brain_mask=True,
                 return_mask=True, csv_path=None, img_size=224):
        
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.use_brain_mask = use_brain_mask
        self.return_mask = return_mask
        self.mask_extractor = BrainMaskExtractor()
        self.img_size = img_size

        # Load data
        self.data = self._load_data()
        self.classes = ['glioma', 'meningioma', 'pituitary']

        # Load features CSV if provided
        self.features_df = None
        if csv_path and os.path.exists(csv_path):
            self.features_df = pd.read_csv(csv_path)

    def _load_data(self):
        """Load all image and mask pairs"""
        data = []

        for class_idx, class_name in enumerate(['glioma', 'meningioma', 'pituitary']):
            class_dir = self.root_dir / class_name

            if not class_dir.exists():
                print(f"Warning: {class_dir} does not exist")
                continue

            # Find all image files
            image_files = sorted([f for f in os.listdir(class_dir)
                                 if 'image' in f and f.endswith('.png')])

            for img_file in image_files:
                mask_file = img_file.replace('image', 'mask')

                img_path = class_dir / img_file
                mask_path = class_dir / mask_file

                if mask_path.exists():
                    data.append({
                        'image_path': str(img_path),
                        'mask_path': str(mask_path),
                        'label': class_idx,
                        'class_name': class_name,
                        'file_id': img_file.split('_')[0]
                    })

        print(f"Loaded {len(data)} samples")
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Load image and tumor mask
        image = cv2.imread(item['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        tumor_mask = cv2.imread(item['mask_path'], cv2.IMREAD_GRAYSCALE)

        # Extract brain mask if needed
        if self.use_brain_mask:
            brain_mask = self.mask_extractor.extract_brain_mask(image)

            # CROP to brain region instead of just masking
            image = self.mask_extractor.crop_to_brain(image, brain_mask, padding=20)
            brain_mask_cropped = self.mask_extractor.crop_to_brain(brain_mask, brain_mask, padding=20)
            tumor_mask = self.mask_extractor.crop_to_brain(tumor_mask, brain_mask, padding=20)

            # Apply mask to remove any remaining background
            image = self.mask_extractor.apply_mask(image, brain_mask_cropped)

        # Resize tumor mask to target size BEFORE transforms
        tumor_mask = cv2.resize(tumor_mask, (self.img_size, self.img_size))
        tumor_mask = torch.from_numpy(tumor_mask).float().unsqueeze(0) / 255.0

        # Apply transforms to image
        if self.transform:
            image = Image.fromarray(image)
            image = self.transform(image)
        else:
            # Default: resize and normalize
            image = cv2.resize(image, (self.img_size, self.img_size))
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        result = {
            'image': image,
            'label': torch.tensor(item['label'], dtype=torch.long),
            'class_name': item['class_name']
        }

        if self.return_mask:
            result['tumor_mask'] = tumor_mask

        return result

# MODEL ARCHITECTURE


class AttentionGuidedResNet50(nn.Module):

    def __init__(self, num_classes=3, pretrained=True, use_attention=True):
        super().__init__()

        # Load pretrained ResNet50
        self.resnet = models.resnet50(pretrained=pretrained)

        # Modify final layer for our classes
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_features, num_classes)

        self.use_attention = use_attention

        # Optional: Add spatial attention module
        if use_attention:
            self.spatial_attention = nn.Sequential(
                nn.Conv2d(2048, 512, kernel_size=3, padding=1),
                nn.BatchNorm2d(512),
                nn.ReLU(),
                nn.Conv2d(512, 1, kernel_size=1),
                nn.Sigmoid()
            )

    def forward(self, x, return_attention=False):
        x = self.resnet.conv1(x)
        x = self.resnet.bn1(x)
        x = self.resnet.relu(x)
        x = self.resnet.maxpool(x)

        x = self.resnet.layer1(x)
        x = self.resnet.layer2(x)
        x = self.resnet.layer3(x)
        features = self.resnet.layer4(x)  # [B, 2048, H/32, W/32]

        attention_map = None
        if self.use_attention:
            attention_map = self.spatial_attention(features)
            features = features * attention_map

        x = self.resnet.avgpool(features)
        x = torch.flatten(x, 1)
        output = self.resnet.fc(x)

        if return_attention:
            return output, attention_map
        return output

# TRAINING FUNCTIONS


class Trainer:

    def __init__(self, model, train_loader, val_loader, device,
                 criterion=None, optimizer=None, scheduler=None,
                 use_attention_loss=False, attention_weight=0.5):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device

        self.criterion = criterion or nn.CrossEntropyLoss()
        self.optimizer = optimizer or torch.optim.Adam(model.parameters(), lr=0.0001)
        self.scheduler = scheduler

        self.use_attention_loss = use_attention_loss
        self.attention_weight = attention_weight

        self.history = {
            'train_loss': [], 'train_acc': [],
            'val_loss': [], 'val_acc': []
        }

    def compute_attention_loss(self, attention_map, tumor_mask):
        
        # Resize attention map to match tumor mask size
        attention_resized = F.interpolate(
            attention_map,
            size=tumor_mask.shape[-2:],
            mode='bilinear',
            align_corners=False
        )

        # Binary cross entropy between attention and tumor mask
        # This encourages the model to focus on tumor regions
        bce_loss = F.binary_cross_entropy(attention_resized, tumor_mask)

        return bce_loss

    def train_epoch(self):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch in self.train_loader:
            images = batch['image'].to(self.device)
            labels = batch['label'].to(self.device)
            tumor_masks = batch.get('tumor_mask', None)

            self.optimizer.zero_grad()

            # Forward pass
            if self.use_attention_loss and tumor_masks is not None:
                outputs, attention_map = self.model(images, return_attention=True)
                tumor_masks = tumor_masks.to(self.device)

                # Classification loss
                cls_loss = self.criterion(outputs, labels)

                # Attention supervision loss
                att_loss = self.compute_attention_loss(attention_map, tumor_masks)

                # Combined loss
                loss = cls_loss + self.attention_weight * att_loss
            else:
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

            # Backward pass
            loss.backward()
            self.optimizer.step()

            # Statistics
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = 100. * correct / total

        return epoch_loss, epoch_acc

    def validate(self):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch in self.val_loader:
                images = batch['image'].to(self.device)
                labels = batch['label'].to(self.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = 100. * correct / total

        return epoch_loss, epoch_acc

    def train(self, num_epochs, save_path='best_model.pth'):
        best_acc = 0.0

        print(f"Training for {num_epochs} epochs...")
        print("-" * 60)

        for epoch in range(num_epochs):
            # Train
            train_loss, train_acc = self.train_epoch()

            # Validate
            val_loss, val_acc = self.validate()

            # Update history
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)

            # Learning rate scheduler
            if self.scheduler:
                self.scheduler.step()

            # Print progress
            print(f"Epoch {epoch+1}/{num_epochs}")
            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
            print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
            print("-" * 60)

            # Save best model
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(self.model.state_dict(), save_path)
                print(f"Model saved with accuracy: {best_acc:.2f}%")

        print(f"\nTraining completed! Best validation accuracy: {best_acc:.2f}%")
        return self.history

# ============================================================================
# 5. EVALUATION AND GRAD-CAM++ VISUALIZATION
# ============================================================================

class ModelEvaluator:

    def __init__(self, model, test_loader, device, class_names):
        self.model = model.to(device)
        self.test_loader = test_loader
        self.device = device
        self.class_names = class_names

    def evaluate(self):
        self.model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in self.test_loader:
                images = batch['image'].to(self.device)
                labels = batch['label'].to(self.device)

                outputs = self.model(images)
                _, predicted = outputs.max(1)

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Classification report
        print("\nClassification Report:")
        print(classification_report(all_labels, all_preds,
                                    target_names=self.class_names))

        # Confusion matrix
        cm = confusion_matrix(all_labels, all_preds)

        return all_preds, all_labels, cm

    def visualize_gradcam(self, image, true_label, pred_label=None,
                         tumor_mask=None, save_path=None):
        self.model.eval()

        # Prepare image
        input_tensor = image.unsqueeze(0).to(self.device)

        # Select target layer (last conv layer)
        target_layers = [self.model.resnet.layer4[-1]]

        # Initialize Grad-CAM++
        cam = GradCAMPlusPlus(model=self.model, target_layers=target_layers)

        # Generate CAM
        targets = [ClassifierOutputTarget(true_label)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]

        # Prepare original image for visualization
        img_np = image.cpu().permute(1, 2, 0).numpy()

        # Denormalize if needed (ImageNet normalization)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = std * img_np + mean
        img_np = np.clip(img_np, 0, 1)

        # Generate CAM visualization
        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        # Create visualization with multiple subplots
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))

        # Original image
        axes[0].imshow(img_np)
        axes[0].set_title('Original MRI', fontsize=12, fontweight='bold')
        axes[0].axis('off')

        # Tumor mask (if available)
        if tumor_mask is not None:
            # Handle different tensor shapes
            mask_np = tumor_mask.cpu().numpy()
            if mask_np.ndim == 3 and mask_np.shape[0] == 1:
                mask_np = mask_np.squeeze(0)  # Remove channel dimension
            axes[1].imshow(mask_np, cmap='hot', interpolation='nearest')
            axes[1].set_title('Ground Truth Tumor Mask', fontsize=12, fontweight='bold')
            axes[1].axis('off')
        else:
            axes[1].axis('off')

        # Grad-CAM++ heatmap
        axes[2].imshow(grayscale_cam, cmap='jet', interpolation='bilinear')
        axes[2].set_title('Grad-CAM++ Heatmap', fontsize=12, fontweight='bold')
        axes[2].axis('off')

        # Overlay
        axes[3].imshow(cam_image)
        title = f'Grad-CAM++ Overlay\nTrue: {self.class_names[true_label]}'
        if pred_label is not None:
            correct = '✓' if pred_label == true_label else '✗'
            title += f'\nPred: {self.class_names[pred_label]} {correct}'
        axes[3].set_title(title, fontsize=12, fontweight='bold')
        axes[3].axis('off')

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')

        plt.show()
        plt.close()

        return cam_image


def main():
    """Main execution function"""

    # Configuration
    DATA_DIR = 'Brain_Dataset'
    CSV_PATH = '.....'
    BATCH_SIZE = 16
    NUM_EPOCHS = 20
    LEARNING_RATE = 0.0001
    IMG_SIZE = 224

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    temp_dataset = BrainTumorDataset(
        root_dir=DATA_DIR,
        transform=None,
        use_brain_mask=True,
        return_mask=True,
        csv_path=CSV_PATH,
        img_size=IMG_SIZE
    )

    train_idx, temp_idx = train_test_split(
        range(len(temp_dataset)), test_size=0.3, random_state=42
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, random_state=42
    )

    train_dataset = BrainTumorDataset(
        root_dir=DATA_DIR,
        transform=train_transform,
        use_brain_mask=True,
        return_mask=True,
        csv_path=CSV_PATH,
        img_size=IMG_SIZE
    )

    val_dataset = BrainTumorDataset(
        root_dir=DATA_DIR,
        transform=val_transform,
        use_brain_mask=True,
        return_mask=True,
        csv_path=CSV_PATH,
        img_size=IMG_SIZE
    )

    test_dataset = BrainTumorDataset(
        root_dir=DATA_DIR,
        transform=val_transform,
        use_brain_mask=True,
        return_mask=True,
        csv_path=CSV_PATH,
        img_size=IMG_SIZE
    )

    train_dataset = torch.utils.data.Subset(train_dataset, train_idx)
    val_dataset = torch.utils.data.Subset(val_dataset, val_idx)
    test_dataset = torch.utils.data.Subset(test_dataset, test_idx)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                             shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)

    print(f"\nDataset splits:")
    print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

    model = AttentionGuidedResNet50(num_classes=3, pretrained=True, use_attention=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    
    trainer = Trainer(model, train_loader, val_loader, device,
                     criterion, optimizer, scheduler)
    history = trainer.train(num_epochs=NUM_EPOCHS, save_path='best_brain_tumor_model.pth')


    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    ax1.plot(history['train_acc'], label='Train Accuracy')
    ax1.plot(history['val_acc'], label='Val Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy (%)')
    ax1.set_title('Model Accuracy')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history['train_loss'], label='Train Loss')
    ax2.plot(history['val_loss'], label='Val Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Model Loss')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

    model.load_state_dict(torch.load('model.pth'))

    evaluator = ModelEvaluator(model, test_loader, device,
                              ['glioma', 'meningioma', 'pituitary'])
    preds, labels, cm = evaluator.evaluate()

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['glioma', 'meningioma', 'pituitary'],
                yticklabels=['glioma', 'meningioma', 'pituitary'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\nGenerating Grad-CAM++ visualizations...")
    model.eval()

    sample_count = 0
    for batch in test_loader:
        if sample_count >= 3:  
            break

        images = batch['image']
        labels = batch['label']
        masks = batch.get('tumor_mask', None)

        with torch.no_grad():
            outputs = model(images.to(device))
            _, preds = outputs.max(1)

        evaluator.visualize_gradcam(
            images[0],
            labels[0].item(),
            preds[0].item(),
            masks[0] if masks is not None else None,
            save_path=f'gradcam_example_{sample_count}.png'
        )

        sample_count += 1

    print("\nPipeline completed successfully!")
    print(f"Models and visualizations saved in current directory.")
    print(f"Best model: best_brain_tumor_model.pth")

if __name__ == "__main__":
    main()